In [ ]:
from _init import *

import os
os.environ['CUDA_VISIBLE_DEVICES'] = '1'

In [ ]:
from copy import deepcopy

from transformers import PreTrainedTokenizerFast, AutoModelForCausalLM

from msnap.utils import common_utils, json_utils, model_utils, tokenizer_utils
from msnap.core import msnap_prompts

In [ ]:
SEED = 42
common_utils.set_seed(SEED)

In [ ]:
work_dir = f'/home/nlpshlee/dev_env/git/repos/msnap'
data_dir = f'{work_dir}/data'

in_file_path = f'{data_dir}/create_contexts/msnap_gpt-5.2_created_contexts.json'
datas = json_utils.load_json(in_file_path)

In [ ]:
model_name = 'Llama-3.2-3B'
# model_name = 'Llama-3.1-8B'

model_name_or_path = f'meta-llama/{model_name}-Instruct'
dtype = 'bfloat16'
device = 'cuda:0'
max_seq_length = 4096
max_new_tokens = 64

model = model_utils.get_model(model_name_or_path, dtype, device=device, is_eval=True)

# 평가/추론 시에는 반드시 'left' 패딩
tokenizer: PreTrainedTokenizerFast = tokenizer_utils.load_tokenizer(model_name_or_path, 'left')

In [ ]:
def get_checked_contexts(answer: str, sorted_keys: list, generated_texts: list, min_size: int):
    checked_keys = []

    for key, generated_text in zip(sorted_keys, generated_texts):
        # print(f'key : {key}, generated_text: {generated_text}')

        if answer.lower() == generated_text.lower():
            checked_keys.append(key)
    
    return (min_size <= len(checked_keys)), checked_keys

In [ ]:
def check_contexts(model: AutoModelForCausalLM, tokenizer: PreTrainedTokenizerFast, datas, out_file_path):
    checked_datas = []
    cnt_q_fact, cnt_q_counter, cnt_q_other = 0, 0, 0
    cnt_checked_fact, cnt_checked_counter, cnt_checked_all = 0, 0, 0

    for i, data in enumerate(datas):
        query = data['question']
        answer_fact = data['answer_fact']
        answer_counter = data['answer_counter']
        contexts_fact: dict = data['contexts_fact']
        contexts_counter: dict = data['contexts_counter']

        sorted_keys = sorted(contexts_fact.keys())
        key_size = len(sorted_keys)

        ''' 1. 프롬프트 생성 '''
        prompts = [msnap_prompts.get_generate_prompt(query)]

        for key in sorted_keys:
            prompts.append(msnap_prompts.get_generate_prompt(query, [contexts_fact[key]]))
        
        for key in sorted_keys:
            prompts.append(msnap_prompts.get_generate_prompt(query, [contexts_counter[key]]))
        
        ''' 2. 모델 결과 생성 '''
        generated_texts = model_utils.get_generated_texts(
            model, tokenizer, device,
            prompts, max_seq_length, max_new_tokens
        )
        
        ''' 3. 모델이 생성한 결과를 실제 정답과 비교 '''
        # 2-1. zero-shot 비교
        checked_q = False
        if generated_texts[0] == answer_fact:
            cnt_q_fact += 1
            # checked_q = True
        elif generated_texts[0] == answer_counter:
            cnt_q_counter += 1
        else:
            cnt_q_other += 1
            checked_q = True

        if checked_q:
            # 2-2. context_fact 비교
            start = 1
            checked_fact, checked_keys_fact = get_checked_contexts(
                answer_fact,
                sorted_keys,
                generated_texts[start:start+key_size],
                3
            )

            # 2-3. context_counter 비교
            start += key_size
            checked_counter, checked_keys_counter = get_checked_contexts(
                answer_counter,
                sorted_keys,
                generated_texts[start:start+key_size],
                1
            )

            if checked_fact: cnt_checked_fact += 1
            if checked_counter: cnt_checked_counter += 1

            if checked_fact and checked_counter:
                cnt_checked_all += 1

                checked_data = deepcopy(data)
                checked_data['contexts_fact'] = {k: contexts_fact[k] for k in checked_keys_fact}
                checked_data['contexts_counter'] = {k: contexts_counter[k] for k in checked_keys_counter}
                checked_datas.append(checked_data)
        
        if (i+1) % 1000 == 0:
            print(f'check_contexts() {i+1} complete.')
            print(f'\tcnt_q_fact : {cnt_q_fact}')
            print(f'\tcnt_q_counter : {cnt_q_counter}')
            print(f'\tcnt_q_other : {cnt_q_other}\n')
            print(f'\tcnt_checked_fact : {cnt_checked_fact}')
            print(f'\tcnt_checked_counter : {cnt_checked_counter}')
            print(f'\tcnt_checked_all : {cnt_checked_all}\n')

    print(f'check_contexts() finished. checked_datas size : {len(checked_datas)}\n')
    json_utils.write_json(checked_datas, out_file_path)
    json_utils.write_jsonl(checked_datas, f'{out_file_path}l')



    print(f'\ncnt_q_fact : {cnt_q_fact}')
    print(f'cnt_q_counter : {cnt_q_counter}')
    print(f'cnt_q_other : {cnt_q_other}\n')
    print(f'cnt_checked_fact : {cnt_checked_fact}')
    print(f'cnt_checked_counter : {cnt_checked_counter}')
    print(f'cnt_checked_all : {cnt_checked_all}')

In [ ]:
out_file_path = f'{data_dir}/msnap_gpt-5.2_checked_{model_name}.json'
check_contexts(model, tokenizer, datas, out_file_path)